# Tool · Compare wrist and index finger speed

**What this notebook does:** It plots the speed of the right wrist and the right index finger together, so you can
see how closely they move. The finger is tracked less reliably than the wrist, so its speed has more spikes.

| You need | You get |
|---|---|
| `Upsampled/Test_Video_processed_data.csv` (from Notebook 2, with `right_wrist` and `right_index` among its KEYPOINTS) | Two interactive plots, and `Upsampled/Test_Video_speed_comparison.csv` with both speeds side by side. |

## Step 1 · Load the tools

In [ ]:
import pandas as pd
import plotly.graph_objects as go

## Step 2 · Settings

In [ ]:
VIDEO_NAME = "Test_Video"

# The stretch of time to show in the first plot, in milliseconds
START_MS = 10_000
END_MS = 100_000

# Finger speeds above this percentile are treated as tracking errors and left out of the second plot
OUTLIER_PERCENTILE = 99

## Step 3 · Load the data

In [ ]:
data = pd.read_csv(f"../Upsampled/{VIDEO_NAME}_processed_data.csv")
speeds = data[["time_ms", "right_wrist_speed", "right_index_speed"]].copy()
speeds.head()

## Step 4 · Plot a stretch of time

To compare the shapes of the two curves, the finger's speed is scaled so that its fastest moment matches the
wrist's fastest moment.

In [ ]:
stretch = speeds[(speeds["time_ms"] >= START_MS) & (speeds["time_ms"] <= END_MS)].copy()
scale = stretch["right_wrist_speed"].max() / stretch["right_index_speed"].max()
stretch["right_index_speed"] *= scale

fig = go.Figure()
fig.add_scatter(x=stretch["time_ms"], y=stretch["right_wrist_speed"], name="right wrist")
fig.add_scatter(x=stretch["time_ms"], y=stretch["right_index_speed"], name="right index finger")
fig.update_layout(title=f"Speed from {START_MS} to {END_MS} ms", xaxis_title="Time (ms)",
                  yaxis_title="Speed")
fig.show()

## Step 5 · Plot the whole video, without the finger's biggest spikes

Moments where the finger speed is above the `OUTLIER_PERCENTILE` are left out, then the finger speed is scaled
as above. The time axis shows minutes:seconds.

In [ ]:
limit = speeds["right_index_speed"].quantile(OUTLIER_PERCENTILE / 100)
cleaned = speeds[speeds["right_index_speed"] <= limit].copy()
scale = cleaned["right_wrist_speed"].max() / cleaned["right_index_speed"].max()
cleaned["right_index_speed"] *= scale
cleaned["time"] = pd.to_datetime(cleaned["time_ms"], unit="ms")  # so the plot can show min:sec

fig = go.Figure()
fig.add_scatter(x=cleaned["time"], y=cleaned["right_wrist_speed"], name="right wrist")
fig.add_scatter(x=cleaned["time"], y=cleaned["right_index_speed"], name="right index finger")
fig.update_layout(title="Speed over the whole video", xaxis_title="Time (min:sec)",
                  yaxis_title="Speed", xaxis_tickformat="%M:%S", xaxis_rangeslider_visible=True)
fig.show()

## Step 6 · Save

In [ ]:
output_path = f"../Upsampled/{VIDEO_NAME}_speed_comparison.csv"
cleaned["time"] = cleaned["time"].dt.strftime("%M:%S.%f").str[:-3]  # minutes:seconds.milliseconds
cleaned.to_csv(output_path, index=False)
print("Saved:", output_path)